# Graph-level t-SNE from reconstructed canonical attention

One t-SNE point is one complete sample graph. All graph preprocessing is delegated to `ResearchSample.structural_features()`: each response token receives a 12-D structural state, then `temporal_summary` reduces the response trajectory to a fixed 36-D graph descriptor (mean/std/slope). Labels are loaded only after t-SNE fitting and are used only for coloring and post-hoc interpretation.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

from descriptors import temporal_summary
from research_dataset import STRUCTURAL_FEATURE_NAMES, ResearchDataset

DATA_ROOT = Path(
    "/share/home/tm902089733300000/a903202310/lys/data/RAGTruth/"
    "model_traces/llama31_8b/test"
)
SAVE_DIR = Path(
    "/share/home/tm902089733300000/a903202310/lys/data/RAGTruth/"
    "visualizations/llama31_8b/canonical_raw_test"
)

DEVICE = "cpu"
VERIFY_HASHES = False
MAX_SAMPLES = None  # e.g. 500 for a quick deterministic run
RANDOM_STATE = 0


In [ ]:
def fit_tsne(matrix, random_state=0):
    if len(matrix) < 3:
        raise ValueError("t-SNE analysis needs at least three samples")
    scaled = StandardScaler().fit_transform(matrix)
    if scaled.shape[1] > 50:
        components = min(50, scaled.shape[0], scaled.shape[1])
        scaled = PCA(n_components=components, random_state=random_state).fit_transform(scaled)
    perplexity = min(30.0, max(2.0, (len(scaled) - 1) / 3.0))
    perplexity = min(perplexity, len(scaled) - 1.0)
    coordinates = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        max_iter=1000,
        random_state=random_state,
    ).fit_transform(scaled)
    return coordinates, perplexity


## Build one fixed descriptor per sample

The canonical CSR is decoded inside `ResearchSample`. No separately built `graphs/` directory is required.


In [ ]:
dataset = ResearchDataset(DATA_ROOT, device=DEVICE, verify_hashes=VERIFY_HASHES)
sample_ids = dataset.sample_ids if MAX_SAMPLES is None else dataset.sample_ids[:MAX_SAMPLES]

descriptor_rows = []
for sample_id in tqdm(sample_ids, desc="graph descriptors"):
    node_states = dataset[sample_id].structural_features()
    descriptor_rows.append(temporal_summary(node_states).cpu().numpy())

descriptor_matrix = np.stack(descriptor_rows)
descriptor_names = np.asarray([
    f"{stat}_{name}"
    for stat in ("mean", "std", "slope")
    for name in STRUCTURAL_FEATURE_NAMES
])
coordinates, perplexity = fit_tsne(descriptor_matrix, RANDOM_STATE)

print("samples:", len(sample_ids))
print("node features:", len(STRUCTURAL_FEATURE_NAMES))
print("graph descriptor shape:", descriptor_matrix.shape)
print(f"t-SNE perplexity: {perplexity:.2f}")


## Evaluation-only coloring

Only now are `positive_runs` loaded. A point is colored hallucinated when the response contains at least one positive token span.


In [ ]:
labels_store = dataset.label_store()
labels = np.asarray([
    int(bool(labels_store.positive_runs(sample_id)))
    for sample_id in sample_ids
], dtype=np.int64)

figure, axis = plt.subplots(figsize=(8, 7), constrained_layout=True)
for value, name, marker in ((0, "Correct sample graph", "o"), (1, "Hallucinated sample graph", "X")):
    mask = labels == value
    axis.scatter(coordinates[mask, 0], coordinates[mask, 1], marker=marker, s=34, alpha=0.75, label=name)
axis.set(title="Graph-level t-SNE", xlabel="t-SNE 1", ylabel="t-SNE 2")
axis.legend()
SAVE_DIR.mkdir(parents=True, exist_ok=True)
figure.savefig(SAVE_DIR / "graph_tsne.png", dpi=200, bbox_inches="tight")
np.savez_compressed(
    SAVE_DIR / "graph_tsne_coordinates.npz",
    sample_id=np.asarray(sample_ids),
    coordinates=coordinates,
    labels=labels,
    descriptor=descriptor_matrix,
    descriptor_names=descriptor_names,
)
plt.show()


## Post-hoc feature differences

This does not affect the embedding; it only helps explain which graph descriptors differ most between correct and hallucinated samples.


In [ ]:
scaled_descriptor = StandardScaler().fit_transform(descriptor_matrix)
difference = scaled_descriptor[labels == 1].mean(axis=0) - scaled_descriptor[labels == 0].mean(axis=0)
order = np.argsort(np.abs(difference))[::-1][:12]
[(descriptor_names[i], float(difference[i])) for i in order]
